# 05 · Collective Algorithms：Centralized、Ring 与 NCCL

**本节问题：** 同样是 AllReduce，不同数据移动方式为什么表现不同？

完成后你应该能够：

- 逐轮解释 ring
- 推导每 rank 的通信量
- 理解教学实现与生产库的差距

前置阅读：[模块 README](../05_tiny_collective/README.md) · [术语表](../docs/concepts/distributed-systems-glossary.md)


## 运行状态卡

默认 `reference` 可在无 GPU 电脑上 Run All。改为 `local` 或 `gpu` 才会启动 runner。

In [ ]:
# 第一处可编辑配置：reference | local | gpu
MODE = "reference"

import sys
from pathlib import Path

notebook_dir = Path("notebooks") if Path("notebooks/_support").is_dir() else Path.cwd()
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

from _support.artifacts import load_artifact
from _support.context import create_context
from _support.plots import bar_chart
from _support.runner import run_command

ctx = create_context("05_collective_algorithms", MODE)
ctx.card()


## 运行前预测

先写下你的预测。不要担心猜错；后面需要指出证据支持或推翻了哪一部分。


In [ ]:
PREDICTION = "我预计……，因为……"
PREDICTION

## 最小观察

这个单元只暴露关键中间状态；正式算法仍来自项目源码。

In [ ]:
world_size = 4
rounds = [('reduce-scatter', i+1) for i in range(world_size-1)] + [('all-gather', i+1) for i in range(world_size-1)]
rounds

## 正确性门与参考证据

读取已提交 JSON；字段缺失时立即停止，不把缺失值解释成 0。

In [ ]:
artifact_path = ctx.repo_root / '05_tiny_collective/results/module05_final_summary.json'
artifact = load_artifact(artifact_path, required=['schema_version', 'artifact_type', 'status', 'correctness'])
print("证据来源：仓库参考结果", artifact_path.relative_to(ctx.repo_root))
print("顶层字段：", sorted(artifact))


In [ ]:
rows = artifact['representative_bus_bandwidth_gbps']['world_4']['67108864_bytes']
for name, value in rows.items(): print(name, value, 'GB/s')
print('边界：', artifact['boundary'])

In [ ]:
values_by_name = artifact['representative_bus_bandwidth_gbps']['world_4']['67108864_bytes']
bar_chart(list(values_by_name), list(values_by_name.values()), title='4 GPU / 64 MiB AllReduce', ylabel='bus bandwidth (GB/s)')

## 本地/正式实验

命令使用参数列表在独立子进程中执行，日志和产物只写入 `_runs/`。

In [ ]:
commands = {
    "local": [sys.executable, '05_tiny_collective/benchmarks/run_correctness.py', '--config', '05_tiny_collective/configs/cpu_correctness.toml', '--output', str(ctx.output_dir / 'correctness.json')],
    "gpu": [sys.executable, '05_tiny_collective/benchmarks/run_gpu_comparison.py', '--config', '05_tiny_collective/configs/gpu_comparison.toml', '--raw-directory', str(ctx.output_dir / 'raw'), '--output', str(ctx.output_dir / 'comparison.json')],
}
command = commands.get(ctx.mode)
if ctx.mode == "reference":
    print("reference 模式：只读已提交证据，不启动实验。")
elif command is None:
    print("本节需要额外环境准备；请使用上方链接中的 Terminal 流程。")
else:
    result = run_command(
        command,
        cwd=ctx.repo_root,
        output_dir=ctx.output_dir,
        label=f"{ctx.mode}-run",
        timeout_seconds=1200,
    )
    print({"passed": result.passed, "seconds": round(result.elapsed_seconds, 2), "log": result.log_path.name})


## 与预测对照、一般规律与边界

请回答：你的预测哪一部分被支持，哪一部分被推翻？当前证据只适用于哪些硬件、shape、消息大小或软件版本？

完成实验后再阅读[正式报告](../05_tiny_collective/experiments/04_final_report.md)。正式报告是结论来源，Notebook 只是交互式观察层。

### 检查题

1. correctness 是否先于性能成立？
2. 当前指标的单位、重复方式和来源是什么？
3. `unavailable`、`failed` 与数值 0 为什么不能混为一谈？

下一步：[打开下一章](06_reducer_bucket_overlap.ipynb)。
